In [73]:
# Instalar pacotes necessários
!pip install tensorflow keras-tuner --quiet


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: C:\Users\pietr\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [74]:
# Importações principais
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from keras_tuner.tuners import RandomSearch

import importlib.util
import sys

# Caminho completo para o arquivo
module_path = "../models/modelo_base_tuner.py"  # ajuste se necessário

# Nome arbitrário para o módulo
module_name = "modelo_base_tuner"

# Cria a especificação do módulo
spec = importlib.util.spec_from_file_location(module_name, module_path)
modelo_base_tuner = importlib.util.module_from_spec(spec)
sys.modules[module_name] = modelo_base_tuner
spec.loader.exec_module(modelo_base_tuner)

# Agora você pode usar:
build_model = modelo_base_tuner.build_model

In [75]:
# Gerar dados de exemplo (substitua pelo seu dataset real, se desejar)
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=1000, 
    n_features=20, 
    n_informative=15, 
    n_classes=2, 
    random_state=42
)


In [76]:
# Divisão em treino e teste e normalização
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# Configuração do Keras Tuner para ajustar learning rate
input_dim = X_train.shape[1]

import os
import shutil
import tensorflow as tf

# Limpa o backend do TensorFlow
tf.keras.backend.clear_session()

# Cria um novo caminho isolado para o tuner
tuner_path = "C:/Users/pietr/Desktop/Resultados/"  # fora do projeto, evita conflitos

# Garante que o diretório seja removido se já existir
if os.path.exists(tuner_path):
    try:
        shutil.rmtree(tuner_path)
        print("Tuner antigo removido com sucesso.")
    except Exception as e:
        print("Erro ao apagar tuner antigo:", e)


from keras_tuner.tuners import RandomSearch

tuner = RandomSearch(
    lambda hp: build_model(hp, input_dim),
    objective="val_accuracy",
    max_trials=10,
    executions_per_trial=1,
    directory=tuner_path,
    project_name="modelo_base_fresh"
)


# Ajustar input_dim dinamicamente
for trial in tuner.oracle.get_space().space:
    if trial.name == "input_dim":
        trial.default = input_dim

# Executar busca
tuner.search(X_train, y_train, epochs=10, validation_split=0.2)

# Avaliação do melhor modelo
best_model = tuner.get_best_models(num_models=1)[0]
test_loss, test_acc = best_model.evaluate(X_test, y_test)
print(f"Acurácia no teste: {test_acc:.4f}")

Tuner antigo removido com sucesso.


FailedPreconditionError: C:/Users/pietr/Desktop/Semestre 02/Aprendizado de Máquina/Projeto02/TrabalhoFinal_Parte2/notebooks is not a directory